In [15]:
import numpy as np
import pandas as pd
import yaml
from sqlalchemy import create_engine

#### Conexion con las bases de datos

In [ ]:
config = {
    'host': 'localhost',       
    'port': , 
    'database': 'mensajeria',  
    'user': 'postgres',            
    'password':"",
}

In [19]:
with open("../config.yml", "r") as f:
    config = yaml.safe_load(f)

config_source = config["mensajeria_bd"]
config_dw = config["etl_mensajeria"]

In [20]:
url_source = (
    f"{config_source['drivername']}://"
    f"{config_source['user']}:{config_source['password']}@"
    f"{config_source['host']}:{config_source['port']}/"
    f"{config_source['dbname']}"
)

url_dw = (
    f"{config_dw['drivername']}://"
    f"{config_dw['user']}:{config_dw['password']}@"
    f"{config_dw['host']}:{config_dw['port']}/"
    f"{config_dw['dbname']}"
)

engine_source = create_engine(url_source)
engine_dw = create_engine(url_dw)

In [21]:
# Extraer novedades
df = pd.read_sql_table(
    "mensajeria_novedadesservicio",
    engine_source
)

df.head()

,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7


In [22]:
# Ver columnas
df.columns

Index(['id', 'fecha_novedad', 'tipo_novedad_id', 'descripcion', 'servicio_id',
       'es_prueba', 'mensajero_id'],
      dtype='str')

In [23]:
# Transformar

fecha = pd.to_datetime(
    df["fecha_novedad"]
)

hecho = pd.DataFrame()

hecho["id_novedad_servicio"] = df["id"]

hecho["id_novedad"] = df["tipo_novedad_id"]

hecho["id_mensajero"] = df["mensajero_id"]

hecho["id_tiempo"] = (
    fecha.dt.strftime("%Y%m%d")
    .astype(int)
)

hecho["id_hora"] = fecha.dt.hour

hecho["cod_servicio"] = df["servicio_id"]

hecho["cantidad_novedades"] = 1

hecho.head()

,id_novedad_servicio,id_novedad,id_mensajero,id_tiempo,id_hora,cod_servicio,cantidad_novedades
0,4,1,7,20231130,5,51,1
1,5,1,7,20231130,5,51,1
2,6,1,7,20231130,5,51,1
3,7,1,7,20231130,5,51,1
4,8,1,7,20231130,5,51,1


#### Validar

In [24]:
hecho.info()

<class 'pandas.DataFrame'>
RangeIndex: 5208 entries, 0 to 5207
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   id_novedad_servicio  5208 non-null   int64
 1   id_novedad           5208 non-null   int64
 2   id_mensajero         5208 non-null   int64
 3   id_tiempo            5208 non-null   int64
 4   id_hora              5208 non-null   int32
 5   cod_servicio         5208 non-null   int64
 6   cantidad_novedades   5208 non-null   int64
dtypes: int32(1), int64(6)
memory usage: 264.6 KB


In [25]:
hecho.describe()

,id_novedad_servicio,id_novedad,id_mensajero,id_tiempo,id_hora,cod_servicio,cantidad_novedades
count,5208.000000,5208.000000,5208.000000,5.208000e+03,5208.000000,5208.000000,5208.0
mean,2617.219086,1.252688,28.740207,2.024055e+07,17.166091,16177.291283,1.0
std,1513.837350,0.434595,13.158249,5.684818e+02,3.800360,8074.096821,0.0
min,4.000000,1.000000,2.000000,2.023113e+07,0.000000,29.000000,1.0
25%,1307.750000,1.000000,16.000000,2.024042e+07,15.000000,9708.750000,1.0
50%,2609.500000,1.000000,29.000000,2.024061e+07,17.000000,16938.000000,1.0
75%,3914.250000,2.000000,41.000000,2.024072e+07,20.000000,23248.000000,1.0
max,5250.000000,2.000000,83.000000,2.024083e+07,23.000000,28467.000000,1.0


In [26]:
hecho.isnull().sum()

id_novedad_servicio    0
id_novedad             0
id_mensajero           0
id_tiempo              0
id_hora                0
cod_servicio           0
cantidad_novedades     0
dtype: int64

#### Verificar fechas

In [27]:
hecho["id_tiempo"].min()

np.int64(20231130)

In [28]:
hecho["id_tiempo"].max()

np.int64(20240831)